## In this notebook we'll learn about Wrapper-method of Feature Selection

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("2_winequalityN.csv")
df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


## Perform some data analysis

In [3]:
df.shape

(6497, 13)

In [4]:
df["quality"].value_counts()

quality
6    2836
5    2138
7    1079
4     216
8     193
3      30
9       5
Name: count, dtype: int64

In [5]:
# df.info()

In [6]:
# df.describe()

In [7]:
df.isnull().sum()   # will fill these null values after train_test_split using SimpleImputer

type                     0
fixed acidity           10
volatile acidity         8
citric acid              3
residual sugar           2
chlorides                2
free sulfur dioxide      0
total sulfur dioxide     0
density                  0
pH                       9
sulphates                4
alcohol                  0
quality                  0
dtype: int64

In [8]:
print("Before Duplicate values: ",df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("After Duplicate values: ",df.duplicated().sum())

Before Duplicate values:  1168
After Duplicate values:  0


## LabelEncode our Categorical column

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["type"] = le.fit_transform(df["type"])
# df["type"]

In [10]:
## Transforming our target column for Logistic Regression

df["quality"] = df["quality"].apply(lambda x : 1 if x>6 else 0)  # below 6 wine quality is not good above 6 wine quality is good 

In [11]:
# splitting the data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df["quality"], test_size=0.2, random_state=1)

print("Training data shape: ",X_train.shape)
print("Testing data shape: ",X_test.shape)

Training data shape:  (4263, 12)
Testing data shape:  (1066, 12)


In [12]:
# filling missing values

cols = X_train.columns
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=cols)
X_test = pd.DataFrame(imputer.transform(X_test), columns=cols)

In [13]:
# Scale the data

from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Baseline Model

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy with all columns: ",accuracy_score(y_test, y_pred))

Accuracy with all columns:  0.8189493433395872


In [15]:
from sklearn.model_selection import cross_val_score

train_cv_score = np.mean(cross_val_score(model, X_train, y_train, cv=10, scoring="accuracy"))
test_score = model.score(X_test,y_test)

print("Training CV Score before Feature Selection: ",train_cv_score)
print("Test Score before Feature Selection: ",test_score)

Training CV Score before Feature Selection:  0.8257094479444975
Test Score before Feature Selection:  0.8189493433395872


# Now Feature Selection using Wrapper method

## Exhaustive Feature Selection

In [16]:
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

model_for_efs = LogisticRegression()

efs = EFS(model_for_efs, max_features=4, scoring="accuracy", cv=5, n_jobs=-1)

sel_for_efs = efs.fit(X_train, y_train)

Features: 793/793

In [17]:
print(f"Best Score: {sel_for_efs.best_score_:.2f}")

Best Score: 0.83


In [18]:
print(f"Best Features: {sel_for_efs.best_feature_names_}")

Best Features: ('5', '6', '10', '11')


In [19]:
## Apply Transformation 

X_train_sel1 = sel_for_efs.transform(X_train)
X_test_sel1 = sel_for_efs.transform(X_test)

In [20]:
model = LogisticRegression()
model.fit(X_train_sel1, y_train)

train_cv_score = np.mean(cross_val_score(model, X_train_sel1, y_train, cv=10, scoring="accuracy"))
test_score = model.score(X_test_sel1,y_test)

print("Training CV Score After Exhaustive Feature Selection: ",train_cv_score)
print("Test Score After Exhaustive Feature Selection: ",test_score)

Training CV Score After Exhaustive Feature Selection:  0.8341574034370156
Test Score After Exhaustive Feature Selection:  0.8170731707317073


-  Before Exhaustive Feature Selection columns : 12, Accuracy : 81 
-  After Exhasutive Feature Selection columns : 4, Accuracy : 81

## Now Sequential Forward Selection

In [21]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

model_for_sfs = LogisticRegression()

sfs = SFS(model_for_sfs, k_features="best", forward=True, floating=True, scoring="accuracy", cv=5)
# k_features="best" automatically finds optimal feature count
# floating=True allows removing previously added features if it improves the score

sel_for_sfs = sfs.fit(X_train, y_train)

In [28]:
print("Best Features for Sequential Forward Selection: ",sel_for_sfs.k_feature_idx_)

Best Features for Sequential Forward Selection:  (4, 5, 6, 10, 11)


In [23]:
X_train_sel2 = sel_for_sfs.transform(X_train)
X_test_sel2 = sel_for_sfs.transform(X_test)

In [24]:
model = LogisticRegression()
model.fit(X_train_sel2, y_train)

train_cv_score = np.mean(cross_val_score(model, X_train_sel2, y_train, cv=10, scoring="accuracy"))
test_score = model.score(X_test_sel2,y_test)

print("Training CV Score After Sequential Forward Selection: ",train_cv_score)
print("Test Score After Sequential Forward Selection: ",test_score)

Training CV Score After Sequential Forward Selection:  0.8329842442633945
Test Score After Sequential Forward Selection:  0.8170731707317073


-  Before Sequential Forward Selection columns : 12, Accuracy : 81 
-  After Sequential Forward Selection columns : 5, Accuracy : 81

## Now Sequential Backward Elimination

In [27]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS2

model_for_sfs2 = LogisticRegression()

sfs2 = SFS2(model_for_sfs2, k_features="best", forward=False, floating=True, scoring="accuracy", cv=5)

sel_for_sfs2 = sfs2.fit(X_train, y_train)

In [31]:
print("Best Features for Sequential Backward Elimination: ",sel_for_sfs2.k_feature_idx_)

Best Features for Sequential Backward Elimination:  (1, 4, 5, 6, 9, 10, 11)


In [32]:
X_train_sel3 = sel_for_sfs2.transform(X_train)
X_test_sel3 = sel_for_sfs2.transform(X_test)